[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tymworld/DROPPS/blob/v1.0.0/examples/colab/04_DROPPS_1_0_Phase_Separation_Analysis.ipynb)

# DROPPS 1.0 Phase Separation Analysis

Analyse the **elongated-box NVT trajectory**, never the compact-box NPT
trajectory. The notebook verifies that pressure coupling is disabled before
running the analysis chain emphasized in the manuscript: density profiles;
residue contact maps and residue-class statistics; chain size and local angle;
intra/inter-chain distances; MSD; and spontaneous-condensation assembly size,
composition, radius of gyration, asphericity, and ellipticity. The notebook
also exposes the auxiliary 1.0 RMSD command.

Select groups explicitly to avoid interactive prompts. Default group numbering
is `System = 0`, followed by molecule types in TOP order. Use the trajectory
preparation notebook to create and inspect custom NDX groups.


In [ ]:
#@title Install DROPPS 1.0
#@markdown The default route installs the immutable wheel attached to the
#@markdown public GitHub `v1.0.0` release. Upload remains available for offline use.
installation_source = "Install from the DROPPS v1.0.0 GitHub release" #@param ["Install from the DROPPS v1.0.0 GitHub release", "Upload the DROPPS 1.0 wheel", "Install from another wheel URL"]
wheel_url = "https://github.com/tymworld/DROPPS/releases/download/v1.0.0/dropps-1.0-py3-none-any.whl" #@param {type:"string"}
accelerator_dependencies = "auto" #@param ["auto", "cuda12", "cuda13", "latest"]

import importlib.metadata
import re
import shutil
import subprocess
import sys
from pathlib import Path


def detect_install_extra():
    if accelerator_dependencies != "auto":
        return accelerator_dependencies
    if shutil.which("nvidia-smi"):
        probe = subprocess.run(
            ["nvidia-smi"], text=True, capture_output=True, check=False
        ).stdout
        match = re.search(r"CUDA Version:\s*(\d+)", probe)
        if match and int(match.group(1)) >= 13:
            return "cuda13"
        return "cuda12"
    return "latest"


if installation_source.startswith("Upload"):
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "This upload form is intended for Google Colab. Set installation_source "
            "to the URL option when running elsewhere."
        ) from exc
    uploaded = files.upload()
    wheel_candidates = [Path(name) for name in uploaded if name.endswith(".whl")]
    if len(wheel_candidates) != 1:
        raise ValueError("Upload exactly one DROPPS 1.0 .whl file.")
    wheel_target = str(wheel_candidates[0].resolve())
else:
    if not wheel_url.strip():
        raise ValueError("Provide the published DROPPS 1.0 wheel URL.")
    wheel_target = wheel_url.strip()

extra = detect_install_extra()
if wheel_target.startswith(("https://", "http://")):
    package_spec = f"dropps[{extra}] @ {wheel_target}"
else:
    package_spec = f"{wheel_target}[{extra}]"
print(f"Installing {package_spec}")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        package_spec,
    ],
    check=True,
)

version = importlib.metadata.version("dropps")
if version != "1.0":
    raise RuntimeError(f"Expected DROPPS 1.0, but installed {version}.")
subprocess.run(["dps", "--version"], check=True)


In [ ]:
#@title Shared notebook helpers
import hashlib
import json
import os
import shlex
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

COMMAND_LOG = []


def run_command(arguments, *, cwd=None, input_text=None, check=True):
    command = [str(value) for value in arguments]
    print("$", shlex.join(command))
    result = subprocess.run(
        command,
        cwd=None if cwd is None else str(cwd),
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    COMMAND_LOG.append(
        {
            "time_utc": datetime.now(timezone.utc).isoformat(),
            "cwd": str(Path(cwd or Path.cwd()).resolve()),
            "command": command,
            "returncode": result.returncode,
        }
    )
    if check and result.returncode:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {shlex.join(command)}"
        )
    return result


def dps(*arguments, cwd=None, input_text=None, check=True):
    return run_command(
        ["dps", *arguments], cwd=cwd, input_text=input_text, check=check
    )


def reset_task_directory(path):
    path = Path(path).resolve()
    if not path.name.startswith("dropps_"):
        raise ValueError(f"Refusing to reset unexpected directory: {path}")
    if path.exists():
        import shutil

        shutil.rmtree(path)
    path.mkdir(parents=True)
    return path


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract_zip(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        bundle.extractall(destination)


def find_unique(root, basename):
    matches = [path for path in Path(root).rglob(basename) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one file named {basename!r} below {root}, found {len(matches)}."
        )
    return matches[0]


def make_zip(paths, output_path, *, base=None):
    output_path = Path(output_path)
    base = Path(base or output_path.parent).resolve()
    with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for source in sorted({Path(path).resolve() for path in paths}):
            if source == output_path.resolve() or not source.is_file():
                continue
            try:
                arcname = source.relative_to(base)
            except ValueError:
                arcname = Path(source.name)
            bundle.write(source, arcname)
    print(f"Wrote {output_path} ({output_path.stat().st_size / 1024:.1f} KiB)")
    return output_path


def download(path):
    try:
        from google.colab import files
    except ImportError:
        print(f"Result available at {Path(path).resolve()}")
    else:
        files.download(str(path))


def optional_time_arguments(start, end, interval):
    arguments = []
    for option, value in (("-b", start), ("-e", end), ("-dt", interval)):
        if str(value).strip():
            arguments.extend([option, str(value).strip()])
    return arguments


def read_xvg(path):
    legends = {}
    metadata = {}
    rows = []
    with Path(path).open(encoding="utf-8") as stream:
        for raw in stream:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("@"):
                re_module = __import__("re")
                match = re_module.search(r's(\d+)\s+legend\s+"(.*)"', line)
                if match:
                    legends[int(match.group(1)) + 1] = match.group(2)
                for key, pattern in {
                    "title": r'@\s+title\s+"(.*)"',
                    "xlabel": r'@\s+xaxis\s+label\s+"(.*)"',
                    "ylabel": r'@\s+yaxis\s+label\s+"(.*)"',
                }.items():
                    label_match = re_module.search(pattern, line)
                    if label_match:
                        metadata[key] = label_match.group(1)
                continue
            rows.append([float(value) for value in line.split()])
    data = np.asarray(rows, dtype=float)
    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(f"No plottable data in {path}")
    return data, legends, metadata


def plot_xvg(path, *, title=None, output=None):
    data, legends, metadata = read_xvg(path)
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for column in range(1, data.shape[1]):
        ax.plot(data[:, 0], data[:, column], label=legends.get(column, f"series {column}"))
    ax.set_title(title or metadata.get("title", Path(path).stem))
    ax.set_xlabel(metadata.get("xlabel", "x / time"))
    ax.set_ylabel(metadata.get("ylabel", "value"))
    if data.shape[1] > 2:
        ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    output_path = Path(output) if output is not None else Path(path).with_suffix(".png")
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    print(f"Saved figure: {output_path}")
    plt.show()
    return fig, output_path


In [ ]:
#@title Upload or reuse a workflow bundle
input_source = "Upload ZIP or individual files" #@param ["Upload ZIP or individual files", "Reuse files from an earlier notebook in this runtime"]
reuse_directory = "/content/dropps_trajectory" #@param {type:"string"}

import shutil

BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
INPUT_ROOT = reset_task_directory(BASE / "dropps_uploaded_inputs")

if input_source.startswith("Upload"):
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload this bundle in Colab or choose the reuse option.") from exc
    uploaded = files.upload()
    for name in uploaded:
        source = Path(name)
        if source.suffix.lower() == ".zip":
            safe_extract_zip(source, INPUT_ROOT)
        else:
            shutil.copy2(source, INPUT_ROOT / source.name)
else:
    source_root = Path(reuse_directory)
    if not source_root.is_dir():
        raise FileNotFoundError(source_root)
    for source in source_root.rglob("*"):
        if source.is_file() and source.suffix.lower() in {
            ".pdb", ".itp", ".top", ".mdp", ".tpr", ".xtc", ".ndx", ".chk", ".xml", ".json"
        }:
            target = INPUT_ROOT / source.name
            if target.exists() and sha256(target) != sha256(source):
                raise ValueError(f"Duplicate filename with different content: {source.name}")
            shutil.copy2(source, target)

print("Available files:")
for path in sorted(INPUT_ROOT.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(INPUT_ROOT), f"({path.stat().st_size / 1024:.1f} KiB)")


## 1 Resolve files and analysis selections


In [ ]:
#@title Input filenames and index groups
run_input_filename = "slab_nvt.tpr" #@param {type:"string"}
trajectory_filename = "processed.xtc" #@param {type:"string"}
index_filename = "analysis.ndx" #@param {type:"string"}
reference_group = 1 #@param {type:"integer"}
two_component_system = False #@param {type:"boolean"}
selection_group = 2 #@param {type:"integer"}
density_center_group = 0 #@param {type:"integer"}
slab_axis = "z"
start_time_ns = "" #@param {type:"string"}
end_time_ns = "" #@param {type:"string"}
sampling_interval_ns = "" #@param {type:"string"}

TPR = find_unique(INPUT_ROOT, run_input_filename)
TRAJECTORY = find_unique(INPUT_ROOT, trajectory_filename)

import zipfile

if not zipfile.is_zipfile(TPR):
    raise ValueError("Expected a portable DROPPS 1.0 TPR v2 file.")
with zipfile.ZipFile(TPR) as archive:
    tpr_parameters = json.loads(archive.read("parameters.json"))
if tpr_parameters.get("pcoulp"):
    raise ValueError(
        "Phase-coexistence analysis requires slab_nvt.tpr (pcoulp=False), "
        "not the compact-box NPT run input."
    )
try:
    NDX = find_unique(INPUT_ROOT, index_filename)
except FileNotFoundError:
    NDX = None

BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
WORKDIR = reset_task_directory(BASE / "dropps_analysis")
if NDX is None:
    NDX = WORKDIR / "analysis.ndx"
    result = dps(
        "make_ndx", "-s", TPR, "-o", NDX,
        cwd=WORKDIR, input_text="q\n", check=False,
    )
    if not NDX.is_file():
        raise RuntimeError(
            f"make_ndx did not create {NDX}; subprocess status was {result.returncode}."
        )
dps("check", "-s", TPR, "-f", TRAJECTORY, "-n", NDX, cwd=WORKDIR)
print("Verified analysis ensemble: NVT (pcoulp = False)")

TIME_ARGS = optional_time_arguments(
    start_time_ns, end_time_ns, sampling_interval_ns
)
ANALYSIS_STATUS = []

def run_analysis(label, arguments):
    try:
        dps(*arguments, cwd=WORKDIR)
    except Exception as exc:
        ANALYSIS_STATUS.append({"analysis": label, "status": "failed", "error": str(exc)})
        print(f"[{label}] FAILED: {exc}")
        return False
    ANALYSIS_STATUS.append({"analysis": label, "status": "complete"})
    return True


## 2 Phase behavior

`density` can output each component separately while centering on a shared
reference density. `contact` supports a global cutoff or residue-specific
σ-scaled cutoff. `cstat` reproduces the residue-class aggregation used for
interaction interpretation in the manuscript.


In [ ]:
#@title Required z-axis density profile and PNG figure
density_axis = "z"
density_type = "mass" #@param ["mass", "charge"]
density_bin_width_nm = 0.05 #@param {type:"number"}
density_center_mode = "frame" #@param ["frame", "block", "none"]
density_blocks = 5 #@param {type:"integer"}
dense_phase_threshold = 0.5 #@param {type:"number"}

if density_axis != "z" or slab_axis != "z":
    raise ValueError("Phase-coexistence density analysis must use the z axis.")
calculate_groups = [reference_group]
if two_component_system:
    calculate_groups.append(selection_group)
arguments = [
    "density", "-s", TPR, "-f", TRAJECTORY, "-n", NDX,
    "-o", "density.xvg", "-x", "z", "-tp", density_type,
    "--bin-width", density_bin_width_nm,
    "-selfit", density_center_group, "-sel", *calculate_groups,
    "--center-mode", density_center_mode, "--blocks", density_blocks,
    "-t", dense_phase_threshold, *TIME_ARGS,
]
density_ok = run_analysis("density", arguments)
if not density_ok:
    raise RuntimeError("Required z-axis density analysis failed.")
_, DENSITY_FIGURE = plot_xvg(
    WORKDIR / "density.xvg",
    title=f"{density_type.title()} density along z",
    output=WORKDIR / "density_z.png",
)


In [ ]:
#@title Contact maps and contact-number time series
run_contacts = True #@param {type:"boolean"}
contact_cutoff_scheme = "global" #@param ["global", "residue"]
global_contact_cutoff_nm = 0.7 #@param {type:"number"}
residue_cutoff_multiplier = 1.2 #@param {type:"number"}
remove_intra_diagonals = 2 #@param {type:"integer"}
contact_use_pbc = True #@param {type:"boolean"}
average_intra_chain_maps = True #@param {type:"boolean"}

contact_ok = False
if run_contacts:
    arguments = [
        "contact", "-s", TPR, "-f", TRAJECTORY, "-n", NDX,
        "-ref", reference_group,
        "-sel", selection_group if two_component_system else reference_group,
        "-cs", contact_cutoff_scheme,
        "-c", global_contact_cutoff_nm, "-cm", residue_cutoff_multiplier,
        "-rd", remove_intra_diagonals, "-otype", "dat",
        "-orr", "contact_reference_reference.dat",
        "-or", "contact_reference_intra.dat",
        "-otrr", "contact_number_reference_reference.xvg",
        *TIME_ARGS,
    ]
    if two_component_system:
        arguments.extend(
            [
                "-sel", selection_group,
                "-ors", "contact_reference_selection.dat",
                "-otrs", "contact_number_reference_selection.xvg",
            ]
        )
    if contact_use_pbc:
        arguments.append("-pbc")
    if average_intra_chain_maps:
        arguments.append("-intraavg")
    contact_ok = run_analysis("contact", arguments)

    if contact_ok:
        for path in sorted(WORKDIR.glob("contact_*.dat")):
            matrix = np.loadtxt(path)
            fig, ax = plt.subplots(figsize=(5.2, 4.4))
            image = ax.imshow(matrix, origin="lower", aspect="auto", cmap="magma")
            ax.set(title=path.stem, xlabel="selection residue", ylabel="reference residue")
            fig.colorbar(image, ax=ax, label="contact value")
            fig.tight_layout()
            figure_path = WORKDIR / f"{path.stem}.png"
            fig.savefig(figure_path, dpi=220, bbox_inches="tight")
            print("Saved figure:", figure_path)
            plt.show()


In [ ]:
#@title Residue-class contact statistics with dps cstat
run_contact_statistics = True #@param {type:"boolean"}
contact_grouping_scheme = "HPST_SC" #@param ["HPST", "AHCP", "HCP", "HPST_SC", "AHCP_SC", "HCP_SC"]
contact_aggregation = "sum" #@param ["sum", "average"]

if run_contact_statistics:
    if not contact_ok:
        print("Contact statistics skipped because no contact map was generated in this run.")
    else:
        map_name = (
            "contact_reference_selection.dat"
            if two_component_system
            else "contact_reference_reference.dat"
        )
        selected = selection_group if two_component_system else reference_group
        run_analysis(
            "cstat",
            [
                "cstat", "-m", map_name, "-s", TPR, "-n", NDX,
                "-ref", reference_group, "-sel", selected,
                "-o", "contact_statistics.xlsx", "-gr",
                "-gs", contact_grouping_scheme, "-ag", contact_aggregation,
            ],
        )


## 3 Chain conformation and dynamics


In [ ]:
#@title Radius of gyration and backbone angles
run_gyrate = True #@param {type:"boolean"}
run_angle = True #@param {type:"boolean"}
gyrate_histogram_bin_width_nm = 0.1 #@param {type:"number"}
conformational_use_pbc = True #@param {type:"boolean"}

if run_gyrate:
    arguments = [
        "gyrate", "-s", TPR, "-f", TRAJECTORY, "-n", NDX,
        "-sel", reference_group, "-oa", "gyrate_average.xvg",
        "-ov", "gyrate_per_chain.xvg", "-oh", "gyrate_histogram.xvg",
        "-bw", gyrate_histogram_bin_width_nm, *TIME_ARGS,
    ]
    if conformational_use_pbc:
        arguments.append("-pbc")
    if run_analysis("gyrate", arguments):
        plot_xvg(WORKDIR / "gyrate_average.xvg", title="Mean chain radius of gyration")

if run_angle:
    arguments = [
        "angle", "-s", TPR, "-f", TRAJECTORY, "-n", NDX,
        "-sel", reference_group, "-ot", "angle_time.xvg",
        "-or", "angle_by_residue.xvg", "-ors", "angle_statistics.xvg",
        *TIME_ARGS,
    ]
    if conformational_use_pbc:
        arguments.append("-pbc")
    if run_analysis("angle", arguments):
        plot_xvg(WORKDIR / "angle_by_residue.xvg", title="Backbone angle by residue")


In [ ]:
#@title Intra- and inter-chain distances
run_intra_chain_distance = False #@param {type:"boolean"}
intra_pair_group_ids = "" #@param {type:"string"}
run_inter_chain_distance = False #@param {type:"boolean"}
inter_reference_group = 0 #@param {type:"integer"}
inter_selection_group = 0 #@param {type:"integer"}
distance_use_pbc = True #@param {type:"boolean"}

if run_intra_chain_distance:
    pair_groups = [int(value) for value in intra_pair_group_ids.replace(",", " ").split()]
    if not pair_groups:
        raise ValueError("Provide one or more NDX groups containing one bead pair per chain.")
    arguments = [
        "idist", "-s", TPR, "-f", TRAJECTORY, "-n", NDX,
        "-sel", *pair_groups, "-ot", "idist_time.xvg",
        "-op", "idist_pairs.xvg", "-ops", "idist_statistics.xvg",
        *TIME_ARGS,
    ]
    if distance_use_pbc:
        arguments.append("-pbc")
    run_analysis("idist", arguments)

if run_inter_chain_distance:
    arguments = [
        "odist", "-s", TPR, "-f", TRAJECTORY, "-n", NDX,
        "-ref", inter_reference_group, "-sel", inter_selection_group,
        "-oa", "odist_average.xvg", "-ov", "odist_all_pairs.xvg",
        *TIME_ARGS,
    ]
    if distance_use_pbc:
        arguments.append("-pbc")
    run_analysis("odist", arguments)


In [ ]:
#@title Mean-square displacement and RMSD
run_msd = True #@param {type:"boolean"}
msd_group = 1 #@param {type:"integer"}
msd_dimensions = "xyz" #@param ["xyz", "xy", "yz", "xz", "x", "y", "z"]
run_rmsd = False #@param {type:"boolean"}
rmsd_selection_expression = "(mol SCAFFOLD)" #@param {type:"string"}
rmsd_fit_mode = "molecule" #@param ["none", "molecule", "selection"]
rmsd_output_mode = "molecule" #@param ["molecule", "selection"]
rmsd_mass_weighted = True #@param {type:"boolean"}

if run_msd:
    arguments = [
        "msd", "-s", TPR, "-f", TRAJECTORY, "-n", NDX,
        "-sel", msd_group, "-t", msd_dimensions,
        "-o", "msd.xvg", *TIME_ARGS,
    ]
    if run_analysis("msd", arguments):
        plot_xvg(WORKDIR / "msd.xvg", title="Mean-square displacement")

if run_rmsd:
    arguments = [
        "rmsd", "-s", TPR, "-f", TRAJECTORY, "-n", NDX,
        "-o", "rmsd.xvg", "--select", rmsd_selection_expression,
        "--fit-mode", rmsd_fit_mode, "--output-mode", rmsd_output_mode,
        *TIME_ARGS,
    ]
    if rmsd_mass_weighted:
        arguments.append("--mass-weighted")
    if run_analysis("rmsd", arguments):
        plot_xvg(WORKDIR / "rmsd.xvg", title="Molecular RMSD")


## 4 Spontaneous-condensation assembly analysis

A cluster is a maximal set of chains connected through intermolecular bead
contacts. Tune the bead cutoff and the minimum contact count to the model.
The molecule-fraction and shape cutoff excludes very small clusters from
composition and morphology statistics.


In [ ]:
#@title Cluster formation, composition, and shape
run_assembly = True #@param {type:"boolean"}
assembly_reference_group = 0 #@param {type:"integer"}
assembly_contact_cutoff_nm = 0.7 #@param {type:"number"}
contacts_to_connect_chains = 5 #@param {type:"integer"}
large_cluster_size_cutoff = 10 #@param {type:"integer"}
assembly_use_pbc = True #@param {type:"boolean"}

if run_assembly:
    selection_groups = [reference_group]
    if two_component_system:
        selection_groups.append(selection_group)
    arguments = [
        "assembly", "-s", TPR, "-f", TRAJECTORY, "-n", NDX,
        "-ref", assembly_reference_group, "-sel", *selection_groups,
        "-c", assembly_contact_cutoff_nm, "-t", contacts_to_connect_chains,
        "-mfc", large_cluster_size_cutoff,
        "-cn", "assembly_cluster_number.xvg",
        "-cs", "assembly_largest_size.xvg",
        "-csd", "assembly_size_distribution.xvg",
        "-mf", "assembly_molecule_fraction.xvg",
        "-rgl", "assembly_largest_rg.xvg",
        "-asp", "assembly_asphericity.xvg",
        "-elp", "assembly_ellipticity.xvg",
        *TIME_ARGS,
    ]
    if assembly_use_pbc:
        arguments.append("-pbc")
    if run_analysis("assembly", arguments):
        plot_xvg(WORKDIR / "assembly_cluster_number.xvg", title="Assembly count")
        plot_xvg(WORKDIR / "assembly_largest_size.xvg", title="Largest assembly size")


## 5 Analysis manifest and results archive


In [ ]:
#@title Summarize and download
manifest = {
    "notebook": "04_DROPPS_1_0_Phase_Separation_Analysis.ipynb",
    "dropps_version": "1.0",
    "source_tpr": {"name": TPR.name, "sha256": sha256(TPR)},
    "source_trajectory": {"name": TRAJECTORY.name, "sha256": sha256(TRAJECTORY)},
    "source_index": {"name": NDX.name, "sha256": sha256(NDX)},
    "verified_ensemble": "NVT",
    "slab_axis": slab_axis,
    "groups": {
        "reference": reference_group,
        "selection": selection_group if two_component_system else None,
        "density_center": density_center_group,
    },
    "time_window_ns": {
        "start": start_time_ns,
        "end": end_time_ns,
        "interval": sampling_interval_ns,
    },
    "analysis_status": ANALYSIS_STATUS,
    "commands": COMMAND_LOG,
}
manifest_path = WORKDIR / "analysis_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
export_files = [path for path in WORKDIR.iterdir() if path.is_file()]
archive = make_zip(
    export_files,
    WORKDIR / "DROPPS_1_0_analysis_results.zip",
    base=WORKDIR,
)
print(json.dumps(ANALYSIS_STATUS, indent=2))
download(archive)
